# 第一讲交互实验：Tokenization 与 Byte-level BPE

本 Notebook 是第一讲的动手实验册。建议先阅读 [第一讲统一讲义](../../lectures/01_overview_tokenization.md)，再按顺序执行每个 Cell。

## 使用方式

- 建议从仓库根目录启动 Jupyter：`jupyter lab`，再打开本文件。
- 本讲实验只使用 Python 标准库与仓库中的教学代码。
- 每个实验先预测结果，再执行代码；改动参数后，重新运行相关 Cell。
- Notebook 当前不保存预制输出。请使用 **Restart Kernel and Run All Cells** 验证完整执行流程。

In [ ]:
from pathlib import Path
import sys

# 允许从仓库根目录或 notebooks/lecture_01/ 目录启动 Jupyter。
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    (path for path in candidates if (path / "code/lecture_01/tokenization_demo.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("没有找到仓库根目录；请从仓库中启动 Jupyter。")

sys.path.insert(0, str(ROOT / "code" / "lecture_01"))
from tokenization_demo import ByteTokenizer, CharacterTokenizer, ToyBPE, merge_pair

print(f"仓库根目录：{ROOT}")
print("教学代码导入成功。")

## 实验 1：字符、Unicode 码点与 UTF-8 字节

**先预测：** 中文字符和 emoji 的 `len(text)` 与 UTF-8 字节数是否相同？组合字符 `é` 又会怎样？

In [ ]:
samples = ["Hi", "你", "🌍", "e\u0301"]

for text in samples:
    code_points = [ord(char) for char in text]
    utf8_bytes = list(text.encode("utf-8"))
    print(
        f"{text!r}: 码点数={len(text)}, "
        f"码点={code_points}, UTF-8 字节数={len(utf8_bytes)}, "
        f"bytes={utf8_bytes}"
    )

**思考：** 字节值只有 0–255 共 256 种，为什么不代表一段文本最多只能有 256 个字节？请分别解释 UTF-8、byte token 与 BPE。

## 实验 2：观察一次非重叠 pair 合并

序列 `[256, 256, 256]` 中，pair `(256, 256)` 有两个候选位置，但两个位置重叠。预测从左向右执行一次合并后会得到什么。

In [ ]:
sequence = [256, 256, 256]
merged = merge_pair(sequence, pair=(256, 256), new_id=257)

print("合并前：", sequence)
print("合并后：", merged)
assert merged == [257, 256]

## 实验 3：训练两轮 Toy BPE

我们用 `ababab` 作为训练语料。每一轮选择当前语料中最常见的相邻 pair，建立新 token，并保存合并顺序。

In [ ]:
bpe = ToyBPE.train(["ababab"], num_merges=2)

print("学到的合并规则：")
for (left, right), new_id in bpe.merges:
    print(f"  ({left}, {right}) -> {new_id}: {bpe.vocab[new_id]!r}")

encoded = bpe.encode("ababab")
print("编码结果：", encoded)
print("解码结果：", bpe.decode(encoded))

assert encoded == [257, 256]
assert bpe.decode(encoded) == "ababab"

## 实验 4：改变 merge budget，观察 token 数

这不是性能 benchmark，只观察固定文本在不同合并预算下的编码长度。注意：合并规则由训练语料学习，因此训练语料也会影响结果。

In [ ]:
train_text = "ababab"
sample_text = "abababab"

for merge_budget in range(6):
    tokenizer = ToyBPE.train([train_text], num_merges=merge_budget)
    ids = tokenizer.encode(sample_text)
    print(
        f"merge_budget={merge_budget}: "
        f"词表大小={len(tokenizer.vocab)}, token 数={len(ids)}, ids={ids}"
    )

## 实验 5：多语言往返检查

往返检查验证 `decode(encode(text)) == text`。它是必要的正确性检查，但不能证明 tokenizer 的压缩效果、速度或模型质量。

In [ ]:
test_samples = [
    "",
    "Hello, world!",
    "你好，世界",
    "🌍🙂",
    "e\u0301",
    "ababab",
]

char_tokenizer = CharacterTokenizer()
byte_tokenizer = ByteTokenizer()
bpe_tokenizer = ToyBPE.train(["ababab", "abab", "abac"], num_merges=4)

for tokenizer_name, tokenizer in [
    ("CharacterTokenizer", char_tokenizer),
    ("ByteTokenizer", byte_tokenizer),
    ("ToyBPE", bpe_tokenizer),
]:
    for text in test_samples:
        restored = tokenizer.decode(tokenizer.encode(text))
        assert restored == text, (tokenizer_name, text, restored)
    print(f"{tokenizer_name}: {len(test_samples)} 个样例全部通过")

## 实验总结：请先独立回答

1. 为什么 token ID 不是语义数值？
2. 为什么 BPE 训练时的 merge 顺序必须被保存？
3. 为什么统计 pair 的次数与实际能合并的次数可能不同？
4. 增加 merge budget 后 token 数发生了什么变化？这个观察能否直接证明推理延迟同比下降？
5. 这份 Notebook 哪些内容适合保留在可复用的 `.py` 中？为什么？

参考：[讲义](../../lectures/01_overview_tokenization.md) · [教学实现](../../code/lecture_01/tokenization_demo.py) · [单元测试](../../tests/lecture_01/test_tokenization_demo.py)